# Phase 7 — Stage B: universal warmup

This stage jointly trains the restoration model across the listed restoration tasks. For the first 50 epochs it routes through two spatial experts with logit jitter σ=0.1, then switches to top-1 routing. Stage B writes atomic `latest.pt` checkpoints every 100 optimizer steps and at each epoch to Drive, plus step logs, per-task loss/PSNR/SSIM matrices, expert load, entropy activity, coherence, and trainer status. Rerunning the training cell with the same output directory automatically resumes at the saved batch. DIV2K is stored in Drive so it remains available after a Colab runtime ends. No restored/intermediate images are generated or archived.

The acquisition cell prepares DIV2K, Flickr2K, CBSD68, SOTS, and Outdoor-Rain under Drive. DIV2K use is restricted to academic research. Flickr2K's EDSR archive and the CBSD68 community mirror (`https://github.com/clausmichele/CBSD68-dataset`) have public-source download attempts; if unavailable, supply a Drive archive/folder. SOTS is distributed from the [RESIDE dataset page](https://sites.google.com/view/reside-dehaze-datasets/reside-v0); access/download packaging varies. Outdoor-Rain does not identify a unique source in the roadmap, so select the intended corpus and provide an authorized direct URL or Drive archive/file ID. Real paired rows can be supplied with `image_path,low_quality_path,task` CSV manifests.


## Data sources (updated 2026-11-03)

The original `cv.snu.ac.kr` EDSR host is dead, so the notebook now uses verified
alternatives: Flickr2K from the Hugging Face `yangtao9009` HR zip (10.85 GB,
downloaded only if you add `flickr2k` to `DATASETS`), CBSD68 from GitHub, DIV2K
from ETH Zurich, SOTS from the Kaggle `balraj98` mirror, Outdoor-Rain/deraining
from the Hugging Face `dronefreak/Rain13K` paired repository (Restormer's
standard deraining protocol), and Urban100 from the Hugging Face
`eugenesiow` HR tar.gz (129 MB, Stage C benchmark only).

Run the cells in order: setup -> Kaggle credentials -> download -> manifests ->
training. The Kaggle cell prompts for your username + API key with masked
input (nothing is stored inside this notebook file); to avoid retyping each
session, save Colab Secrets named `KAGGLE_USERNAME` and `KAGGLE_API_KEY` and
the cell picks them up automatically. If Drive space is tight, keep
`DATASETS` trimmed to one new corpus per session - downloaded corpora are
reused and training resumes from `latest.pt`.

In [ ]:
from pathlib import Path
import subprocess
import sys

from google.colab import drive
drive.mount('/content/drive')
REPO = Path('/content/SparseDict')
DATA_ROOT = Path('/content/drive/MyDrive/SparseDict/datasets')
RUN_DIR = Path('/content/drive/MyDrive/SparseDict/phase7/stage_b')
STAGE_A = Path('/content/drive/MyDrive/SparseDict/phase7/stage_a/pretrained_dictionaries.pth')
RUN_DIR.mkdir(parents=True, exist_ok=True)
if not REPO.exists():
    subprocess.run(['git', 'clone', '--branch', 'master', 'https://github.com/kalpeshjp89/SparseDict.git', str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', 'master'], check=True)
sys.path.insert(0, str(REPO))
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime in Colab before training.')

In [ ]:
# @title 2) Kaggle credentials (needed for the SOTS download)
# Kaggle token setup - paste ONCE per runtime. On kaggle.com go to
# Account -> API -> "Create New Token"; the downloaded kaggle.json contains
# exactly two values: username and key. Input below is masked with getpass
# and is never written into the notebook file.
import json
import os
import subprocess
import sys
from pathlib import Path


def _colab_secret(name: str) -> str:
    try:
        from google.colab import userdata

        return userdata.get(name)
    except Exception:
        return ""


KAGGLE_USERNAME_INPUT = _colab_secret("KAGGLE_USERNAME")
KAGGLE_API_KEY_INPUT = _colab_secret("KAGGLE_API_KEY")
if not (KAGGLE_USERNAME_INPUT and KAGGLE_API_KEY_INPUT):
    import getpass

    KAGGLE_USERNAME_INPUT = getpass.getpass("Kaggle username: ").strip()
    KAGGLE_API_KEY_INPUT = getpass.getpass("Kaggle API key: ").strip()

KAGGLE_READY = False
if not (KAGGLE_USERNAME_INPUT and KAGGLE_API_KEY_INPUT):
    print(
        "Kaggle credentials skipped. The `sots` kaggle: source below will "
        "fail until this cell is completed with username + key."
    )
else:
    kaggle_dir = Path.home() / ".kaggle"
    kaggle_dir.mkdir(parents=True, exist_ok=True)
    token_path = kaggle_dir / "kaggle.json"
    token_path.write_text(
        json.dumps({"username": KAGGLE_USERNAME_INPUT, "key": KAGGLE_API_KEY_INPUT})
    )
    token_path.chmod(0o600)
    os.environ["KAGGLE_CONFIG_DIR"] = str(kaggle_dir)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "kaggle"]
    ).check_returncode()
    result = subprocess.run(
        [
            "kaggle",
            "datasets",
            "files",
            "balraj98/synthetic-objective-testing-set-sots-reside",
        ],
        capture_output=True,
        text=True,
    )
    KAGGLE_READY = result.returncode == 0
    print(
        "kaggle auth check:",
        "OK" if KAGGLE_READY else (result.stderr or result.stdout).strip()[:300],
    )
    if KAGGLE_READY:
        print(
            "Kaggle credentials installed at ~/.kaggle/kaggle.json. If the "
            "check above reports a 403/permission error instead, accept the "
            "dataset once: open the dataset page on Kaggle while signed in "
            "and agree to its access rule, then rerun this cell."
        )

In [ ]:
# @title 3) Download the datasets (Drive), one corpus per session if space is tight
# Small -> large order. Already-downloaded folders on Drive are reused, so a
# rerun only fetches what is missing. Keep 'flickr2k' (10.85 GB HR zip) out
# of the list unless this session has room for it.
DATASETS = ['cbsd68', 'sots', 'outdoor-rain', 'div2k']
DOWNLOAD_PUBLIC_DATASETS = True
# DIV2K is academic-research-only; explicitly accept its terms to download.
ACCEPT_DIV2K_TERMS = False
SOURCE_URLS = {
    # SOTS dehazing mirror (requires the Kaggle credentials cell to pass).
    'sots': 'kaggle:balraj98/synthetic-objective-testing-set-sots-reside',
    # Outdoor-Rain / deraining training pairs (Restormer's Rain13K protocol,
    # spread HF repository downloaded with huggingface_hub).
    'outdoor-rain': 'hf-dataset:dronefreak/Rain13K',
}
DRIVE_ARCHIVES = {}  # e.g. {'sots': '/content/drive/MyDrive/datasets/SOTS.zip'}
DRIVE_FILE_IDS = {}  # e.g. {'cbsd68': 'Google-Drive-file-id'}
command = [sys.executable, str(REPO / 'scripts' / 'prepare_phase7_datasets.py'),
           '--root', str(DATA_ROOT)]
for dataset in DATASETS:
    command.extend(['--dataset', dataset])
if DOWNLOAD_PUBLIC_DATASETS:
    command.append('--download-public')
if ACCEPT_DIV2K_TERMS:
    command.append('--accept-div2k-license')
for dataset, path in DRIVE_ARCHIVES.items():
    command.extend(['--drive-archive', f'{dataset}={path}'])
for dataset, file_id in DRIVE_FILE_IDS.items():
    command.extend(['--drive-file-id', f'{dataset}={file_id}'])
for dataset, url in SOURCE_URLS.items():
    command.extend(['--source-url', f'{dataset}={url}'])
subprocess.run(command, cwd=REPO, check=True)

In [ ]:
from scripts.prepare_phase7_datasets import resolve_clean_image_dir
# Override a detected folder if a dataset uses a different clean/GT layout.
DATASET_PATH_OVERRIDES = {}  # e.g. {'sots': DATA_ROOT / 'SOTS' / 'outdoor' / 'GT'}
DATASET_DIRS = {
    name: Path(DATASET_PATH_OVERRIDES[name]) if name in DATASET_PATH_OVERRIDES
    else resolve_clean_image_dir(name, DATA_ROOT)
    for name in DATASETS
}
IMAGE_DIRS = list(DATASET_DIRS.values())
# The Kaggle SOTS mirror stores GT as separate indoor/outdoor halves. The
# resolver above returns only the first clean folder, so append any further
# gt/clear folders that exist next to it (skipping ones already selected).
_SOTS_ROOT = DATA_ROOT / 'SOTS'
_SOTS_IMAGE_SUFFIXES = {'.png', '.jpg', '.jpeg', '.bmp'}
if _SOTS_ROOT.is_dir():
    for sub in sorted(
        p for p in _SOTS_ROOT.rglob('*')
        if p.is_dir() and p.name.casefold() in {'gt', 'clear', 'target', 'ground_truth'}
        and any(f.suffix.lower() in _SOTS_IMAGE_SUFFIXES for f in p.rglob('*'))
    ):
        if sub not in IMAGE_DIRS:
            IMAGE_DIRS.append(sub)

print('Training image folders:', DATASET_DIRS)
PAIRED_MANIFESTS = []
# Example: PAIRED_MANIFESTS.append(Path('/content/drive/MyDrive/datasets/paired_train.csv'))
if not all(path.is_dir() for path in IMAGE_DIRS):
    raise FileNotFoundError('Download DIV2K or configure IMAGE_DIRS first.')

TRAIN_MANIFEST = RUN_DIR / 'train.csv'
VALIDATION_MANIFEST = RUN_DIR / 'validation.csv'
TASKS = 'sr2,sr3,sr4,denoise15,denoise25,denoise50,derain,desnow,dehaze,lowlight,jpeg'
manifest_command = [
    sys.executable, str(REPO / 'scripts' / 'make_universal_manifest.py'),
    '--train-output', str(TRAIN_MANIFEST), '--validation-output', str(VALIDATION_MANIFEST),
    '--validation-fraction', '0.1', '--seed', '2027', '--tasks', TASKS
]
for directory in IMAGE_DIRS:
    manifest_command.extend(['--image-dir', str(directory)])
for paired_manifest in PAIRED_MANIFESTS:
    manifest_command.extend(['--paired-manifest', str(paired_manifest)])
subprocess.run(manifest_command, cwd=REPO, check=True)
if not STAGE_A.is_file():
    raise FileNotFoundError(f'Run Stage A first: {STAGE_A}')

In [ ]:
command = [
    sys.executable, str(REPO / 'experiments' / 'train_universal.py'),
    '--phase', 'warmup', '--train-manifest', str(TRAIN_MANIFEST),
    '--validation-manifest', str(VALIDATION_MANIFEST), '--init-checkpoint', str(STAGE_A),
    '--output-dir', str(RUN_DIR), '--epochs', '50', '--warmup-epochs', '50',
    '--steps-per-epoch', '100', '--patch-size', '128',
    '--validation-samples-per-task', '8',
    '--checkpoint-every-steps', '100', '--device', 'cuda'
]
print(' '.join(command))
subprocess.run(command, cwd=REPO, check=True)
print('Metrics and status:', RUN_DIR / 'metrics.jsonl', RUN_DIR / 'metrics_by_task.csv', RUN_DIR / 'trainer_status.json')